# Aula 3 - Tool Calling

## Versão 3.0 do projeto: um assistente que pode pedir ações

Nesta aula, vamos descrever funções que a aplicação pode executar quando a pergunta exigir uma ação.

Ao final, você deve conseguir:

- descrever uma ferramenta com nome, descrição e schema;
- identificar um `tool_call` na resposta do modelo;
- validar e executar a função correspondente na aplicação;
- devolver o resultado em uma mensagem `tool`;
- fazer uma segunda chamada para produzir a resposta final.

## 1. Configuração do ambiente

Instale as dependências pelo terminal, seguindo o `README.md` desta pasta. Copie `env.example` para `.env` e preencha `GROQ_API_KEY`. Nunca coloque a chave diretamente neste arquivo nem faça commit do `.env`.

In [ ]:
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

## 2. Descreva as ferramentas

O modelo não recebe uma função Python diretamente. Ele recebe um contrato com nome, descrição e parâmetros em JSON Schema.

In [ ]:
# TODO: crie uma lista chamada `tools` com pelo menos duas funções.
# Cada item deve ter o formato:
# {
#     "type": "function",
#     "function": {
#         "name": "nome_da_funcao",
#         "description": "Explique quando usar a ferramenta.",
#         "parameters": {
#             "type": "object",
#             "properties": {},
#             "required": [],
#         },
#     },
# }

## 3. Implemente as funções

As ferramentas desta aula são funções locais. Elas simulam uma consulta a dados de estudo e um cálculo, sem depender de outra API.

In [ ]:
STUDY_CONTENT = {
    "1": {"tema": "LLM APIs", "resumo": "Chamadas, mensagens, tokens e ausência de memória automática."},
    "2": {"tema": "Context Engineering", "resumo": "Seleção e organização do contexto antes da chamada."},
    "3": {"tema": "Tool Calling", "resumo": "Contratos de ferramentas, pedidos estruturados e execução pela aplicação."},
}

# TODO: implemente `consultar_aula(aula)` usando STUDY_CONTENT.
# Retorne um dicionário com found, aula, tema e resumo.

# TODO: implemente `calcular_media(notas)`.
# Valide uma lista de números entre 0 e 10 e retorne media, quantidade e aprovado.

## 4. Crie o mapa de execução

O nome vindo do modelo nunca deve virar código executado arbitrariamente. A aplicação mantém um mapa explícito entre nomes permitidos e funções conhecidas.

In [ ]:
AVAILABLE_TOOLS = {
    "consultar_aula": consultar_aula,
    "calcular_media": calcular_media,
}

print(sorted(AVAILABLE_TOOLS))

## 5. Faça a primeira chamada

Envie `tools` junto com as mensagens. Com `tool_choice="auto"`, o modelo pode responder diretamente ou solicitar uma função.

In [ ]:
query = "[DIGITE AQUI SUA PERGUNTA]"
messages = [
    {
        "role": "system",
        "content": (
            "Você é um assistente de estudos. Responda diretamente quando souber. "
            "Use uma ferramenta quando a pergunta exigir uma consulta ou cálculo."
        ),
    },
    {"role": "user", "content": query},
]

first_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
    temperature=0,
)

## 6. Inspecione o pedido do modelo

A resposta pode conter texto, `tool_calls` ou os dois. Observe o nome da função, os argumentos e o identificador da chamada.

In [ ]:
assistant_message = first_response.choices[0].message
print("Texto: ", assistant_message.content)
print("Tool calls: ", assistant_message.tool_calls)

## 7. Execute as ferramentas

A aplicação valida o nome recebido, converte os argumentos de JSON e chama uma função do mapa permitido.

In [ ]:
# TODO: percorra `assistant_message.tool_calls`.
# Para cada chamada:
# 1. leia name e arguments;
# 2. verifique se name está em AVAILABLE_TOOLS;
# 3. use json.loads(arguments);
# 4. execute a função;
# 5. guarde o resultado junto com call.id.

In [ ]:
# Exemplo de estrutura para executar com segurança:
# tool_results = []

# for call in assistant_message.tool_calls or []:
#     name = call.function.name
#     if name not in AVAILABLE_TOOLS:
#         raise ValueError(f"Ferramenta não permitida: {name}")

#     arguments = json.loads(call.function.arguments)
#     result = AVAILABLE_TOOLS[name](**arguments)
#     tool_results.append({"id": call.id, "name": name, "result": result})

# print(tool_results)

## 8. Devolva o resultado para o modelo

A aplicação precisa reenviar a mensagem `assistant` com o pedido e acrescentar uma mensagem `tool` para cada chamada. O `tool_call_id` conecta o resultado ao pedido correto.

In [ ]:
# TODO: acrescente ao histórico:
# 1. a mensagem assistant com content e tool_calls;
# 2. uma mensagem tool por resultado, com tool_call_id;

# 3. faça uma segunda chamada sem remover as mensagens anteriores:
    for item in tool_results:
        messages.append({
            "role": "tool",
            "tool_call_id": item["id"],
            "name": item["name"],
            "content": json.dumps(item["result"], ensure_ascii=False),
        })

# 4. salve a resposta final em `final_response`:
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    temperature=0,
)

print(final_response.choices[0].message.content)

## 9. Compare uma resposta direta com uma tool

Faça uma pergunta conceitual e outra que dependa de cálculo ou consulta. Observe quando o modelo responde diretamente e quando inicia o fluxo de ferramenta.

In [ ]:
# TODO: teste uma pergunta que não precise de ferramenta.
# Depois teste uma pergunta como: "Qual é a média de 8, 7 e 9?"
# Compare as respostas e registre qual caminho foi usado.

In [ ]:
# Um exemplo prático de comparação:
# conceptual_response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "O que é uma tool?"}],
#     temperature=0,
# )

# print("RESPOSTA DIRETA")
# print(conceptual_response.choices[0].message.content)
# print("\nFLUXO COM TOOL")
# print("Pergunta:", query)
# print("Pedidos:", [(item["name"], item["result"]) for item in tool_results])
# print("Resposta final:", final_response.choices[0].message.content)

## 10. Checkpoint da V3

A V3 está pronta quando:

- o schema descreve as ferramentas;
- a primeira chamada envia `tools`;
- você inspeciona `tool_calls`;
- o nome recebido é validado;
- a função Python executa;
- a mensagem `tool` usa o `tool_call_id` correto;
- a segunda chamada produz a resposta final;
- a chave continua no `.env`;
- as alterações foram salvas em um commit separado.